In [31]:
import intake
import numpy as np
import polars as pl
from itertools import product
from jetutils.data import extract, flatten_by, standardize
from jetutils.definitions import KAPPA, DATADIR, compute

url = 'https://raw.githubusercontent.com/NCAR/cesm2-le-aws/main/intake-catalogs/aws-cesm2-le.json'

In [25]:
catalog = intake.open_esm_datastore(url)

In [26]:
catalog.df.columns

Index(['variable', 'long_name', 'component', 'experiment', 'forcing_variant',
       'frequency', 'vertical_levels', 'spatial_domain', 'units', 'start_time',
       'end_time', 'path'],
      dtype='object')

In [27]:
vars_ = ["variable", "long_name", "vertical_levels", "spatial_domain"]
pl.from_pandas(catalog.df).unique(vars_).sort("vertical_levels", "variable").to_numpy()

array([[None, None, 'atm', 'historical', 'smbb', 'static', None,
        'global', None, None, None,
        's3://ncar-cesm2-lens/atm/static/grid.zarr'],
       [None, None, 'ice', 'historical', 'smbb', 'static', None,
        'global_ocean', None, None, None,
        's3://ncar-cesm2-lens/ice/static/grid.zarr'],
       ['FLNS', 'net longwave flux at surface', 'atm', 'historical',
        'cmip6', 'daily', '1.0', 'global', 'W/m2', '1850-01-01 12:00:00',
        '2014-12-31 12:00:00',
        's3://ncar-cesm2-lens/atm/daily/cesm2LE-historical-cmip6-FLNS.zarr'],
       ['FLNSC', 'clearsky net longwave flux at surface', 'atm',
        'historical', 'cmip6', 'daily', '1.0', 'global', 'W/m2',
        '1850-01-01 12:00:00', '2014-12-31 12:00:00',
        's3://ncar-cesm2-lens/atm/daily/cesm2LE-historical-cmip6-FLNSC.zarr'],
       ['FLUT', 'upwelling longwave flux at top of model', 'atm',
        'historical', 'cmip6', 'daily', '1.0', 'global', 'W/m2',
        '1850-01-01 12:00:00', '2014-1

In [46]:
year = 1999
component = "atm"
frequency = "daily"
forcing_variant = "cmip6"
experiment = "historical" if year < 2016 else "ssp370"
variable = "Z3"

catalog_subset = catalog.search(
    variable=variable,
    component=component,
    experiment=experiment,
    frequency=frequency,
    forcing_variant=forcing_variant,
)
dsets = catalog_subset.to_dataset_dict(
    xarray_open_kwargs={"consolidated": False},
    storage_options={"anon": True}
)
indexers = [component, experiment, frequency, forcing_variant]
indexers = [np.atleast_1d(indexer) for indexer in indexers]
indexers = [[str(idx_) for idx_ in indexer] for indexer in indexers]
indexers = list(product(*indexers))
ensemble_keys = [".".join(indexer) for indexer in indexers]
ds = dsets[ensemble_keys[0]]
ds = standardize(ds)
period = year
season = None
minlon = -80
maxlon = 40
minlat = 15
maxlat = 80
levels = 12
members = "all"
ds = extract(
    ds,
    period=period,
    season=season,
    minlon=minlon,
    maxlon=maxlon,
    minlat=minlat,
    maxlat=maxlat,
    levels=levels,
    members=members,
)
ds = compute(ds, progress_flag=True)


--> The keys in the returned dictionary of datasets are constructed as follows:
	'component.experiment.frequency.forcing_variant'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:04&lt;00:00]</div>

[#                                       ] | 3% Completed | 468.96 ss


KeyboardInterrupt: 

In [ ]:
import xarray

ds = xarray.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    chunks=None,
    storage_options=dict(token='anon'),
)
ar_full_37_1h = ds.sel(time=slice(ds.attrs['valid_time_start'], ds.attrs['valid_time_stop']))